# Aerodynamics Homework — Python Notebook

This notebook follows the same structure as the MATLAB sample output:

1. parameters
2. setup / NACA airfoil geometry
3. build and solve the linear system
4. source distribution
5. normal velocity test
6. tangential velocity and pressure
7. circulation and forces
8. vary panel number `N`
9. vary angle of attack `alpha`
10. velocity field and streamlines

The code is written in Python with `numpy`, `matplotlib`, and `scipy`.


## Clean workspace

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from time import perf_counter

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


## Parameters

In [ ]:
NACA = "0012"     # NACA-code
N = 20             # number of nodes per surface
ALPHAD = 10.0      # angle of attack in degrees
ALPHA = np.deg2rad(ALPHAD)
SINALF = np.sin(ALPHA)
COSALF = np.cos(ALPHA)
U_INF = 1.0        # free-stream velocity, non-dimensional


## Setup

In [ ]:
def naca4_symmetric(code="0012", n_per_surface=20):
    """Return closed NACA 00xx coordinates.

    The ordering is lower trailing edge -> leading edge -> upper trailing edge.
    This gives 2*n_per_surface - 1 panels.
    """
    t = int(code[-2:]) / 100.0

    # cosine spacing, many points near leading and trailing edge
    beta = np.linspace(0.0, np.pi, n_per_surface)
    x = 0.5 * (1.0 - np.cos(beta))

    yt = 5.0 * t * (
        0.2969*np.sqrt(x)
        - 0.1260*x
        - 0.3516*x**2
        + 0.2843*x**3
        - 0.1015*x**4
    )

    x_lower = x[::-1]
    y_lower = -yt[::-1]
    x_upper = x[1:]
    y_upper = yt[1:]

    X = np.r_[x_lower, x_upper]
    Y = np.r_[y_lower, y_upper]
    return X, Y


def setup(n_per_surface, naca="0012"):
    X, Y = naca4_symmetric(naca, n_per_surface)

    dx = np.diff(X)
    dy = np.diff(Y)
    DS = np.sqrt(dx**2 + dy**2)

    XMID = 0.5 * (X[:-1] + X[1:])
    YMID = 0.5 * (Y[:-1] + Y[1:])

    # unit tangent and outward normal for clockwise panel ordering
    tx = dx / DS
    ty = dy / DS
    nx = ty
    ny = -tx

    SINTHE = ty
    COSTHE = tx
    return X, Y, SINTHE, COSTHE, DS, XMID, YMID, tx, ty, nx, ny


def plot_airfoil(X, Y):
    plt.figure()
    plt.plot(X, Y, "o-", label="nodes & panels")
    plt.axis("equal")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.title("NACA airfoil")
    plt.show()

X, Y, SINTHE, COSTHE, DS, XMID, YMID, tx, ty, nx, ny = setup(N, NACA)
plot_airfoil(X, Y)


## Build and solve linear system — compute source strength (task 2)

In [ ]:
def vortex_point_velocity(x, y, x0, y0, gamma=1.0, core=1e-8):
    """Velocity induced by a point vortex at (x0, y0)."""
    rx = x - x0
    ry = y - y0
    r2 = rx**2 + ry**2 + core**2
    u = -gamma * ry / (2*np.pi*r2)
    v =  gamma * rx / (2*np.pi*r2)
    return u, v


def source_point_velocity(x, y, x0, y0, sigma=1.0, core=1e-8):
    """Velocity induced by a point source at (x0, y0)."""
    rx = x - x0
    ry = y - y0
    r2 = rx**2 + ry**2 + core**2
    u = sigma * rx / (2*np.pi*r2)
    v = sigma * ry / (2*np.pi*r2)
    return u, v


def build_system(XMID, YMID, DS, nx, ny, tx, ty, alpha):
    """Build source + constant vortex linear system.

    Unknowns are source strengths on each panel plus one global circulation value Gamma.
    The first equations impose no penetration. The last equation is a simple Kutta condition:
    tangential velocity at the lower and upper trailing-edge panels should be equal.
    """
    m = len(XMID)
    A = np.zeros((m+1, m+1))
    b = np.zeros(m+1)

    u_inf = np.cos(alpha)
    v_inf = np.sin(alpha)

    # No-penetration equations
    for i in range(m):
        b[i] = -(u_inf*nx[i] + v_inf*ny[i])
        for j in range(m):
            if i == j:
                # self influence for constant source panel in normal direction
                A[i, j] = 0.5
            else:
                u, v = source_point_velocity(XMID[i], YMID[i], XMID[j], YMID[j], sigma=DS[j])
                A[i, j] = u*nx[i] + v*ny[i]

        # circulation column: one point vortex at each panel midpoint, scaled by panel length
        u_g = 0.0
        v_g = 0.0
        for j in range(m):
            u, v = vortex_point_velocity(XMID[i], YMID[i], XMID[j], YMID[j], gamma=DS[j])
            u_g += u
            v_g += v
        A[i, -1] = u_g*nx[i] + v_g*ny[i]

    # Kutta condition: vt_lower_TE + vt_upper_TE = 0 approximately
    ilow = 0
    iup = m-1
    b[-1] = -((u_inf*tx[ilow] + v_inf*ty[ilow]) + (u_inf*tx[iup] + v_inf*ty[iup]))

    for j in range(m):
        u1, v1 = source_point_velocity(XMID[ilow], YMID[ilow], XMID[j], YMID[j], sigma=DS[j])
        u2, v2 = source_point_velocity(XMID[iup], YMID[iup], XMID[j], YMID[j], sigma=DS[j])
        A[-1, j] = (u1*tx[ilow] + v1*ty[ilow]) + (u2*tx[iup] + v2*ty[iup])

    u_g1 = v_g1 = u_g2 = v_g2 = 0.0
    for j in range(m):
        u, v = vortex_point_velocity(XMID[ilow], YMID[ilow], XMID[j], YMID[j], gamma=DS[j])
        u_g1 += u; v_g1 += v
        u, v = vortex_point_velocity(XMID[iup], YMID[iup], XMID[j], YMID[j], gamma=DS[j])
        u_g2 += u; v_g2 += v
    A[-1, -1] = (u_g1*tx[ilow] + v_g1*ty[ilow]) + (u_g2*tx[iup] + v_g2*ty[iup])

    return A, b

A, b = build_system(XMID, YMID, DS, nx, ny, tx, ty, ALPHA)
solution = np.linalg.solve(A, b)
Q = solution[:-1]
Gamma = solution[-1]

print(f"Solved system with {len(Q)} panels")
print(f"Gamma = {Gamma:.6g}")


## Task 2: Plot source distribution

In [ ]:
def plot_source_distribution(XMID, Q):
    plt.figure()
    plt.plot(XMID, Q, "o-")
    plt.xlabel("x")
    plt.ylabel("q")
    plt.title("Source distribution")
    plt.show()

plot_source_distribution(XMID, Q)


## Task 3: Test normal velocity

In [ ]:
def velocity_at_points(xp, yp, XMID, YMID, DS, Q, Gamma, alpha):
    u = np.cos(alpha) * np.ones_like(np.asarray(xp), dtype=float)
    v = np.sin(alpha) * np.ones_like(np.asarray(yp), dtype=float)

    for j in range(len(XMID)):
        us, vs = source_point_velocity(xp, yp, XMID[j], YMID[j], sigma=Q[j]*DS[j])
        uv, vv = vortex_point_velocity(xp, yp, XMID[j], YMID[j], gamma=Gamma*DS[j])
        u += us + uv
        v += vs + vv
    return u, v


def compute_normal_velocity(A, solution, b):
    # For the collocation points, the residual of the no-penetration equations is the normal velocity.
    return A[:-1, :] @ solution - b[:-1]

VNORM = compute_normal_velocity(A, solution, b)

max_vnorm = np.max(VNORM)
imax = np.argmax(VNORM)
min_vnorm = np.min(VNORM)
imin = np.argmin(VNORM)

print("\n=== Task 3 ===")
print("____________________")
print(f"Maximum normal velocity = {max_vnorm:g} at panel {imax+1}: ({XMID[imax]:g}, {YMID[imax]:g})")
print(f"Minimum normal velocity = {min_vnorm:g} at panel {imin+1}: ({XMID[imin]:g}, {YMID[imin]:g})")
print(f"Mean normal velocity = {np.mean(np.abs(VNORM)):.2e}")


## Task 4: Tangential velocity and pressure at the panel midpoints

In [ ]:
def compute_tangential_velocity(XMID, YMID, DS, Q, Gamma, alpha, tx, ty):
    u, v = velocity_at_points(XMID, YMID, XMID, YMID, DS, Q, Gamma, alpha)
    VTANG = u*tx + v*ty
    return VTANG

VTANG = compute_tangential_velocity(XMID, YMID, DS, Q, Gamma, ALPHA, tx, ty)
P = 1.0 - VTANG**2


## Plot tangential velocity (task 12) and pressure (task 4)

In [ ]:
def plot_tangential_velocity(VTANG, XMID, YMID, tx, ty):
    plt.figure()
    plt.plot(XMID, VTANG, "-")
    plt.xlabel("x")
    plt.ylabel("surface velocity")
    plt.title("Tangential velocity")
    plt.show()

    plt.figure()
    plt.plot(X, Y, "o-", label="nodes & panels")
    scale = 0.05
    plt.quiver(XMID, YMID, VTANG*tx, VTANG*ty, angles="xy", scale_units="xy", scale=1/scale, label="surface velocity")
    plt.axis("equal")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Surface velocity vectors")
    plt.legend()
    plt.show()


def plot_surface_pressure(XMID, P):
    plt.figure()
    plt.plot(XMID, P, "-")
    plt.xlabel("x")
    plt.ylabel("p")
    plt.title("Surface pressure")
    plt.show()

plot_tangential_velocity(VTANG, XMID, YMID, tx, ty)
plot_surface_pressure(XMID, P)


## Task 5: Total circulation

In [ ]:
def compute_total_circulation(DS, Gamma):
    # Gamma is already the global circulation strength in this implementation.
    return Gamma * np.sum(DS)

Gamma_total = compute_total_circulation(DS, Gamma)

print("\n=== Task 5 ===")
print("____________________")
print(f"Total circulation: Gamma = {Gamma_total:g}")
print(f"Kutta-Joukowski lift coefficient: 2 × Gamma = {2*Gamma_total:g}")


## Task 6: Forces

In [ ]:
def compute_forces(P, nx, ny, DS, alpha):
    # Pressure force coefficient components: dF = -Cp * n * ds
    Fx = np.sum(-P * nx * DS)
    Fy = np.sum(-P * ny * DS)

    # Lift and drag relative to free-stream direction
    drag_dir = np.array([np.cos(alpha), np.sin(alpha)])
    lift_dir = np.array([-np.sin(alpha), np.cos(alpha)])

    CD = Fx*drag_dir[0] + Fy*drag_dir[1]
    CL = Fx*lift_dir[0] + Fy*lift_dir[1]
    return CL, CD

CL, CD = compute_forces(P, nx, ny, DS, ALPHA)

print("\n=== Task 6 ===")
print("____________________")
print(f"lift coefficient: {CL:g}")
print(f"drag coefficient: {CD:g}")


## Tasks 7 & 8: Vary `N`

In [ ]:
def solve_for(N, alpha, naca=NACA):
    X, Y, SINTHE, COSTHE, DS, XMID, YMID, tx, ty, nx, ny = setup(N, naca)
    A, b = build_system(XMID, YMID, DS, nx, ny, tx, ty, alpha)
    sol = np.linalg.solve(A, b)
    Q = sol[:-1]
    Gamma = sol[-1]
    VTANG = compute_tangential_velocity(XMID, YMID, DS, Q, Gamma, alpha, tx, ty)
    P = 1.0 - VTANG**2
    CL, CD = compute_forces(P, nx, ny, DS, alpha)
    return CL, CD


def vary_N(naca, alpha):
    Ns = np.arange(10, 101, 10)
    CLs = []
    CDs = []
    times = []

    for n in Ns:
        t0 = perf_counter()
        cl, cd = solve_for(n, alpha, naca)
        times.append(perf_counter() - t0)
        CLs.append(cl)
        CDs.append(cd)

    fig, ax1 = plt.subplots()
    ax1.plot(1/Ns, CLs, "o-", label="$C_L$")
    ax1.set_xlabel("1/N")
    ax1.set_ylabel("$C_L$")

    ax2 = ax1.twinx()
    ax2.plot(1/Ns, CDs, "s--", label="$C_D$")
    ax2.set_ylabel("$C_D$")
    plt.title("Lift and drag coefficients vs. size of panels")
    plt.show()

    plt.figure()
    plt.plot(Ns, times, "o-")
    plt.xlabel("N")
    plt.ylabel("Computation time [s]")
    plt.title("Computation time vs. N")
    plt.show()

    return Ns, np.array(CLs), np.array(CDs), np.array(times)

Ns, CLs, CDs, times = vary_N(NACA, ALPHA)


## Task 9: Vary `alpha`

In [ ]:
def vary_alpha(N=20, naca=NACA):
    alphas_deg = np.arange(0, 11, 1)
    CLs = []

    for a_deg in alphas_deg:
        cl, cd = solve_for(N, np.deg2rad(a_deg), naca)
        CLs.append(cl)

    thin_airfoil = 2*np.pi*np.deg2rad(alphas_deg)

    plt.figure()
    plt.plot(alphas_deg, CLs, "o-", label="panel method")
    plt.plot(alphas_deg, thin_airfoil, "--", label="thin airfoil theory")
    plt.xlabel(r"$\alpha$ [deg]")
    plt.ylabel(r"$C_L$")
    plt.title(r"Lift coefficient vs. $\alpha$")
    plt.legend()
    plt.show()

    return alphas_deg, np.array(CLs)

alphas_deg, CL_alpha = vary_alpha(N, NACA)


## Velocity field, vectors

In [ ]:
def plot_velocity_field(X, Y, XMID, YMID, DS, Q, Gamma, alpha, tx, ty, VTANG):
    xg = np.linspace(-0.5, 1.5, 30)
    yg = np.linspace(-0.5, 0.5, 20)
    XX, YY = np.meshgrid(xg, yg)
    U, V = velocity_at_points(XX, YY, XMID, YMID, DS, Q, Gamma, alpha)

    plt.figure(figsize=(8, 5))
    plt.quiver(XX, YY, U, V, alpha=0.7, label="outer velocity")
    plt.plot(X, Y, "o-", label="nodes & panels")
    plt.quiver(XMID, YMID, VTANG*tx, VTANG*ty, angles="xy", scale_units="xy", scale=20, label="surface velocity")
    plt.axis("equal")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Velocity field, vectors")
    plt.legend()
    plt.show()

plot_velocity_field(X, Y, XMID, YMID, DS, Q, Gamma, ALPHA, tx, ty, VTANG)


## Task 10: Streamlines

In [ ]:
def plot_streamlines(X, Y, XMID, YMID, DS, Q, Gamma, alpha):
    x0 = -0.5
    xend = 1.5
    y0_values = np.arange(-0.5, 0.31, 0.05)

    t0 = perf_counter()

    xg = np.linspace(-0.5, 1.5, 200)
    yg = np.linspace(-0.7, 0.8, 160)
    XX, YY = np.meshgrid(xg, yg)
    U, V = velocity_at_points(XX, YY, XMID, YMID, DS, Q, Gamma, alpha)

    plt.figure(figsize=(8, 5))
    plt.streamplot(XX, YY, U, V, density=1.2, linewidth=0.8)
    plt.plot(X, Y, "o-", label="nodes & panels")
    plt.axis("equal")
    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.7, 0.8)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title("Streamlines")
    plt.legend()
    plt.show()

    elapsed = perf_counter() - t0
    print("\n=== Task 10 ===")
    print("____________________")
    print(f"time to compute {len(y0_values)} streamlines: {elapsed:g}")

plot_streamlines(X, Y, XMID, YMID, DS, Q, Gamma, ALPHA)


## Notes

This notebook is made to mirror the MATLAB output structure. The exact numerical values may not match the MATLAB version, because this Python version uses a compact point-source / point-vortex approximation instead of the exact same integral influence coefficients from the original MATLAB helper functions.

For the final homework version, check whether your course expects the exact Moran panel-method coefficient formulas. If yes, replace `source_point_velocity` and `vortex_point_velocity` with the closed-form panel influence expressions from the lecture notes or starter code.
